# 🛒 네이버 쇼핑 전체 대표 인기물품 TOP 10 고객군별 데이터 분석

네이버 데이터랩 쇼핑인사이트 API를 통해 수집된 **네이버 쇼핑 전체 10개 대표 분야 인기 물품의 최근 2일치 고객군(기기, 성별, 연령) 클릭 데이터**를 Pandas DataFrame으로 로드하고 탐색합니다.

---
### 📌 분석 대상 10개 대표 물품
- **패션의류**: 원피스
- **패션잡화**: 크록스
- **화장품/미용**: ahc아이크림
- **디지털/가전**: 냉장고
- **가구/인테리어**: 식탁의자
- **출산/육아**: 물티슈
- **식품**: 추석선물세트
- **스포츠/레저**: 텐트
- **생활/건강**: 마스크
- **디지털/IT**: 노트북

## 1. 라이브러리 임포트 및 환경 설정

In [ ]:
import os
import pandas as pd

# 데이터프레임 출력 행/열 수 확대 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 1000)
print('✅ Pandas 버전:', pd.__version__)

## 2. CSV 데이터 파일 로드
`data/` 디렉터리에 저장된 수집 결과 CSV 파일을 자동으로 탐색하여 로드합니다.

In [ ]:
csv_filename = 'shopping_insight_전체쇼핑_TOP10물품_고객군분석_2일치.csv'
candidates = [
    csv_filename,
    os.path.join('..', 'data', csv_filename),
    os.path.join('data', csv_filename),
    os.path.join(r'C:\projects\wepscraping-git\data', csv_filename)
]

target_file = None
for path in candidates:
    if os.path.exists(path):
        target_file = path
        break

if target_file:
    df = pd.read_csv(target_file, encoding='utf-8-sig')
    print(f'✅ 데이터 로드 성공: {target_file}')
    print(f'- 총 행 수: {df.shape[0]}행, 컬럼 수: {df.shape[1]}개')
else:
    raise FileNotFoundError(f'❌ CSV 파일을 찾을 수 없습니다: {csv_filename}')

## 3. 데이터프레임 기본 확인 (`head`, `info`, 고유값)

In [ ]:
# 상위 10개 행 미리보기
df.head(10)

In [ ]:
# 데이터 컬럼 정보 및 결측치 확인
df.info()

In [ ]:
# 수집된 10개 대표 물품 목록 확인
items_df = df[['순번', '카테고리명', '물품검색어']].drop_duplicates().sort_values('순번')
items_df.reset_index(drop=True)

## 4. 기기별(PC vs 모바일) 이용 비중 분석
각 물품별로 모바일과 PC 중 어떤 기기를 통한 유입이 많은지 비교합니다.

In [ ]:
df_device = df[df['분석구분'] == 'DEVICE'].copy()
device_pivot = df_device.pivot_table(
    index=['순번', '카테고리명', '물품검색어'],
    columns='고객군명',
    values='클릭비율지수',
    aggfunc='mean'
).round(2)

device_pivot

## 5. 성별(남성 vs 여성) 이용 비중 분석
각 물품별로 여성과 남성 고객층의 선호도를 분석합니다.

In [ ]:
df_gender = df[df['분석구분'] == 'GENDER'].copy()
gender_pivot = df_gender.pivot_table(
    index=['순번', '카테고리명', '물품검색어'],
    columns='고객군명',
    values='클릭비율지수',
    aggfunc='mean'
).round(2)

gender_pivot

## 6. 연령대별(10대~60대) 선호도 분석
10대부터 60대 이상까지 연령대별 클릭 비율 지수를 피벗 테이블로 비교합니다.

In [ ]:
df_age = df[df['분석구분'] == 'AGE'].copy()
age_columns = ['10대', '20대', '30대', '40대', '50대', '60대 이상']

age_pivot = df_age.pivot_table(
    index=['순번', '카테고리명', '물품검색어'],
    columns='고객군명',
    values='클릭비율지수',
    aggfunc='mean'
).reindex(columns=age_columns).round(2)

age_pivot

## 7. 10개 대표 물품별 클릭비율 통계 요약 (`describe`)
수집된 클릭비율지수의 수치적 분포를 확인합니다.

In [ ]:
df.groupby(['카테고리명', '물품검색어'])['클릭비율지수'].describe().round(2)